In [1]:
!pip install -q sentence-transformers pandas tqdm


In [2]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
DATASET_PATH = "/content/drifted_prompts.json"
OUTPUT_FOLDER = "/content/prompt_similarity_outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [4]:
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [84]:
with open(DATASET_PATH, "r") as f:
    data = json.load(f)

prompts = data["prompts"]

In [85]:
from collections import defaultdict

In [86]:
structured = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))


In [87]:
for item in prompts:
    category = item["category"]
    base_id = item["base_id"]
    drift = item["drift_level"]
    structured[category][base_id][drift].append(item)

print("Dataset structured successfully.")

Dataset structured successfully.


In [88]:
prompts

[{'id': 'reasoning_1_L0',
  'base_id': 'reasoning_1',
  'category': 'reasoning',
  'drift_level': 0,
  'variant': 0,
  'prompt': 'A city is facing frequent power outages due to rising energy demand. Explain how the government should decide between investing in renewable energy infrastructure versus expanding fossil fuel-based plants. Consider economic, environmental, and long-term sustainability factors.'},
 {'id': 'reasoning_1_L1_V1',
  'base_id': 'reasoning_1',
  'category': 'reasoning',
  'drift_level': 1,
  'variant': 1,
  'prompt': 'A city is facing frequent power outages due to rising energy demand. Explain how the government should decide between investing in renewable energy infrastructure versus expanding fossil fuel-baded planta. Consider economic, environmental, and long-term sustainability factors.'},
 {'id': 'reasoning_1_L1_V2',
  'base_id': 'reasoning_1',
  'category': 'reasoning',
  'drift_level': 1,
  'variant': 2,
  'prompt': 'A city is facing frequent power ougages du

In [89]:
def compute_similarity(clean_text, drift_text):
    emb_clean = model.encode(clean_text, normalize_embeddings=True)
    emb_drift = model.encode(drift_text, normalize_embeddings=True)
    return float(cosine_similarity([emb_clean], [emb_drift])[0][0])


In [90]:
import re

def natural_sort_key(text):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', text)]

In [91]:
for category, bases in structured.items():

    print(f"Processing category: {category}")

    base_ids = sorted(bases.keys(), key=natural_sort_key)

    # Prepare dataframe structure
    rows = []

    for drift_level in [1,2,3,4,5]:
        row = {"Drift/Variant": drift_level}

        for base_id in base_ids:

            clean_prompt = bases[base_id][0][0]["prompt"]  # L0

            # Each drift level has 3 variants
            variants = sorted(bases[base_id][drift_level], key=lambda x: x["variant"])

            for v in variants:
                sim = compute_similarity(clean_prompt, v["prompt"])

                col_name = f"{base_id}_V{v['variant']}"
                row[col_name] = round(sim * 100, 2)  # convert to percentage

        rows.append(row)

    df = pd.DataFrame(rows)

    # Save CSV
    save_path = os.path.join(OUTPUT_FOLDER, f"{category}_prompt_similarity.csv")
    df.to_csv(save_path, index=False)

    print(f"Saved → {save_path}")


Processing category: reasoning
Saved → /content/prompt_similarity_outputs/reasoning_prompt_similarity.csv
Processing category: logical
Saved → /content/prompt_similarity_outputs/logical_prompt_similarity.csv
Processing category: classification
Saved → /content/prompt_similarity_outputs/classification_prompt_similarity.csv
Processing category: qna
Saved → /content/prompt_similarity_outputs/qna_prompt_similarity.csv


In [92]:
categories = ["classification", "logical", "qna", "reasoning"]

for category in categories:
    print(category)
    df = pd.read_csv(f"{OUTPUT_FOLDER}/{category}_prompt_similarity.csv")

    # Remove the first column (drift level column)
    values = df.iloc[:, 1:]

    # Compute average cosine similarity per level
    df["average_similarity"] = round(values.mean(axis=1), 2)

    print(df[["Drift/Variant", "average_similarity"]])

classification
   Drift/Variant  average_similarity
0              1               97.19
1              2               94.57
2              3               91.98
3              4               86.94
4              5               78.75
logical
   Drift/Variant  average_similarity
0              1               96.73
1              2               94.84
2              3               88.58
3              4               84.19
4              5               77.46
qna
   Drift/Variant  average_similarity
0              1               96.27
1              2               95.14
2              3               89.26
3              4               82.63
4              5               75.34
reasoning
   Drift/Variant  average_similarity
0              1               96.23
1              2               94.76
2              3               89.22
3              4               86.13
4              5               77.86


In [93]:
!zip -r /content/prompt_similarity_outputs.zip /content/prompt_similarity_outputs


  adding: content/prompt_similarity_outputs/ (stored 0%)
  adding: content/prompt_similarity_outputs/logical_prompt_similarity.csv (deflated 62%)
  adding: content/prompt_similarity_outputs/reasoning_prompt_similarity.csv (deflated 63%)
  adding: content/prompt_similarity_outputs/qna_prompt_similarity.csv (deflated 59%)
  adding: content/prompt_similarity_outputs/classification_prompt_similarity.csv (deflated 67%)
